In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix,
)

# Import Data

In [13]:
ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "social-media-extremism-detection-challenge"

# Jika notebook berada di GEMASTIK-LATIHAN, naik satu folder.
if not DATA_DIR.exists():
    ROOT = ROOT.parent
    DATA_DIR = ROOT / "data" / "social-media-extremism-detection-challenge"

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("\nDistribusi label:")
print(train_df["Extremism_Label"].value_counts(dropna=False))


Train: (2250, 3)
Test: (750, 2)

Distribusi label:
Extremism_Label
EXTREMIST        1149
NON_EXTREMIST    1101
Name: count, dtype: int64


# EDA

In [14]:
assert "Original_Message" in train_df.columns
assert "Extremism_Label" in train_df.columns
assert "Original_Message" in test_df.columns
assert "ID" in test_df.columns
assert train_df["Extremism_Label"].notna().all()


assert digunakan untuk memastikan asumsi tertentu benar.
Misalnya, model mengharapkan kolom Original_Message. Jika nama kolom ternyata Message_Text, program sebaiknya berhenti dan memberi tahu kita bahwa struktur data tidak sesuai, alih-alih gagal jauh di tengah proses.
Prinsipnya adalah gagal lebih awal dengan pesan yang jelas (fail fast).
Dalam proyek kompetisi, pemeriksaan seperti ini membantu mencegah kesalahan akibat dataset yang berbeda format, file yang salah, atau perubahan struktur data.

In [15]:
# Tangani pesan kosong tanpa membuang baris training.
#Baris yang berisi NaN diubah menajdi ""
X = train_df["Original_Message"].fillna("").astype(str)
y = train_df["Extremism_Label"]

X_test = test_df["Original_Message"].fillna("").astype(str)

print("\nJumlah pesan kosong:", int(X.eq("").sum()))
print("Kelas:", sorted(y.unique()))


Jumlah pesan kosong: 1
Kelas: ['EXTREMIST', 'NON_EXTREMIST']


# Split Data

In [3]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training:", len(X_train))
print("Validation:", len(X_valid))

Training: 1800
Validation: 450


# Pemodelan

Model machine learning seperti Logistic Regression tidak dapat langsung menghitung hubungan statistik dari kalimat mentah seperti:
"this message contains harmful language"

Model membutuhkan representasi numerik.
Salah satu pendekatan yang sederhana adalah TF-IDF, singkatan dari Term Frequency–Inverse Document Frequency.

Inti konsepnya: TF-IDF bukan model klasifikasi. TF-IDF adalah cara merepresentasikan teks sebagai fitur; Logistic Regression adalah model yang mempelajari hubungan antara fitur dan label.

TF-IDF bukan pemahaman bahasa seperti manusia. Metode ini tidak otomatis memahami sarkasme, konteks sosial, atau makna tersirat. Namun, ia sering menjadi baseline yang kuat untuk klasifikasi teks.

In [4]:
baseline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,           # Mengubah teks jadi huruf kecil
            ngram_range=(1, 2),       # Bigram: dua token berurutan, misalnya bad message.
            min_df=2,                 # fitur harus muncul pada setidaknya dua dokumen training agar dipertahankan.
            max_features=100_000,     # Membatasi jumlah fitur yang dipertahankan hingga maksimum 100.000, untuk kontrol penggunaan memori dan waktu pelatihan
            sublinear_tf=True,        # Transformasi ini dapat mengurangi dominasi kata yang berulang sangat sering dalam satu dokumen.
        ),
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,             #Batas maksimum iterasi
            class_weight="balanced",   #kelas jarang dapat bobot yang lebih besar
            random_state=42,           #untuk bantu komponen pengacakan
        ),
    ),
])

baseline.fit(X_train, y_train)      #Latih model baseline

valid_pred = baseline.predict(X_valid)  #Untuk uji validasi model paai data X_valid

print("Accuracy:", round(accuracy_score(y_valid, valid_pred), 4))
print("Macro-F1:", round(f1_score(
    y_valid, valid_pred, average="macro"
), 4))

print("\nClassification report:")
print(classification_report(y_valid, valid_pred, zero_division=0))

print("Confusion matrix:")
print(confusion_matrix(
    y_valid,
    valid_pred,
    labels=baseline.classes_,
))
print("Urutan kelas:", baseline.classes_)

Accuracy: 0.7933
Macro-F1: 0.7933

Classification report:
               precision    recall  f1-score   support

    EXTREMIST       0.82      0.77      0.79       230
NON_EXTREMIST       0.77      0.82      0.80       220

     accuracy                           0.79       450
    macro avg       0.79      0.79      0.79       450
 weighted avg       0.79      0.79      0.79       450

Confusion matrix:
[[176  54]
 [ 39 181]]
Urutan kelas: ['EXTREMIST' 'NON_EXTREMIST']


In [5]:
# Latih ulang pada seluruh data berlabel.
baseline.fit(X, y)

# Prediksi data test yang tidak memiliki label.
test_pred = baseline.predict(X_test)

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "Extremism_Label": test_pred,
})

# Pemeriksaan format dasar.
assert submission.columns.tolist() == ["ID", "Extremism_Label"]
assert len(submission) == len(test_df)
assert submission["ID"].equals(test_df["ID"])
assert submission["Extremism_Label"].notna().all()
assert set(submission["Extremism_Label"].unique()).issubset(set(y.unique()))

output_dir = ROOT / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "submission_baseline.csv"
submission.to_csv(output_path, index=False)

print("Submission tersimpan di:", output_path)
print("\nDistribusi prediksi:")
print(submission["Extremism_Label"].value_counts())
print("\nPreview:")
print(submission.head())

Submission tersimpan di: C:\Users\hajar\Documents\gemastik-xix\JupyterProject\outputs\submission_baseline.csv

Distribusi prediksi:
Extremism_Label
NON_EXTREMIST    508
EXTREMIST        242
Name: count, dtype: int64

Preview:
     ID Extremism_Label
0  2251   NON_EXTREMIST
1  2252   NON_EXTREMIST
2  2253   NON_EXTREMIST
3  2254   NON_EXTREMIST
4  2255   NON_EXTREMIST
